# 05: 多分辨率 Leiden 聚类

以多个 Leiden 分辨率运行聚类并对结果进行可视化比较，帮助选择最符合生物学粒度的分群方案。

**上游**：04 产出的嵌入结果
**本 notebook 产出**：
- `obs["leiden_res_{resolution}"]` 列（各分辨率聚类标签）
- 各分辨率着色的 UMAP 图
- 聚类指标对比表与群大小分布图
- `05_clustered_v1.h5ad` checkpoint，供 06 标注使用

**扩展槽**（注释掉的 cell）：任何写出 `obs["{method}_clusters"]` 的聚类方法
均可与 `leiden_res_*` 列共存。ACDC 不是默认依赖，安装后取消注释即可插入。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：04（多方法嵌入），读 `04_embedded_v*.h5ad`
- **下游**：06（多方法注释），产出 `05_clustered_v*.h5ad`

### 为什么要迭代回跑？
聚类分群的分辨率选择直接影响下游注释的质量。如果在 06（注释时发现
某个细胞类型被错误拆分或合并、LLM 判决分歧高）发现问题，可能需要：
- 调整 `RESOLUTIONS` 列表（加更细或更粗的分辨率）
- 换用不同的嵌入（改 `USE_REP` 指向 04 的另一个嵌入）
- 换用 04 的另一个版本（不同嵌入方法/参数组合）

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `04_embedded_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`
   （例如 `05_clustered_v2.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `RESOLUTIONS` 或 `USE_REP`）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为分群质量可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：06 的 `UPSTREAM_PATH` 指向你决定采用的 05 版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"05_clustered"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询"05 有哪些版本？哪些依赖 04_v1？"，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

In [ ]:
# === PARAMS ===

UPSTREAM_PATH = "results/04_embedded_v1.h5ad"
OUTPUT_PATH   = "results/05_clustered_v1.h5ad"

# --- 使用的嵌入 ---
USE_REP = "X_pca_harmony"

# --- 邻居图（支持 Scalar-or-Sweep）---
RECOMPUTE_NEIGHBORS = False         # True = 用下方参数重新计算（与 04 的 k 不同时用）
N_NEIGHBORS = 15                    # 单值 | 列表如 [10, 15, 20, 30] → 对比不同 k 对分群的影响
N_PCS_USE_FOR_NEIGHBORS = None      # None = 使用嵌入全部维度（适合 Harmony/scVI 已降维的输出）
                                    # 设整数（如 30）= 对 X_pca 等高维嵌入截断前 N 维
MIN_CLUSTER_SIZE = 20               # 小于此值的 cluster 统计功效弱，标记 ⚠️

# --- Leiden 分辨率（天然列表 = sweep）---
# 生物学锚点参考：
#   - 胃黏膜 ~5 大类（上皮/免疫/基质/内皮/神经）→ cluster 数 10-15 对应 res ≈ 0.4-0.8
#   - 发现新亚型 → 故意过聚类 res ≥ 1.5，再按 dendrogram 合并
#   - 已知高异质性组织 → 宽范围 sweep [0.2-2.0]
RESOLUTIONS = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]

# --- 聚类稳定性分析 ---
STABILITY_ENABLED = True
STABILITY_N_ITER  = 10              # 子抽样次数
STABILITY_SUBSAMPLE_FRAC = 0.8      # 每次抽 80%

# --- Consensus Clustering ---
CONSENSUS_CLUSTERING = False        # True=在多个分辨率上聚类取 consensus（计算密集，可选）

# --- Marker 基因预览（每个分辨率的 quick sanity check）---
MARKER_PREVIEW_ENABLED = True
MARKER_PREVIEW_N_GENES = 3          # 每个 cluster 显示 top N 基因

# --- 过聚类策略 ---
OVER_CLUSTERING_RES = None          # 非 None 时标记该分辨率为"故意过聚类"
# --- 锚定 marker：验证基本分离格局是否正确（不是正式注释，只是 sanity check）---
SANITY_MARKERS = {
    "Epithelial": "EPCAM",
    "Immune": "PTPRC",
    "Stromal": "VIM",
    "Proliferating": "MKI67",
}


OUTPUT_VERSION = 1
RANDOM_SEED    = 42

In [ ]:
# 确保框架 src/ 在 sys.path 上，CWD 为项目根目录。
# 自动检测两种运行场景：从 notebooks/（Jupyter）还是项目根目录（nbconvert）启动。
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/figures/sweep_05", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

In [ ]:
# 导入（scanpy 原生 API + 框架函数仅在真正有缺口时使用）。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import warnings
from scipy.stats import median_abs_deviation

# 直接从 scorers 模块导入——无回调，在 for 循环中直接调用
from scrna_integration.scorers import clustering_metrics

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

## 加载上游数据

读取 04 产出的嵌入结果，确认数据维度与嵌入键名。
后续所有操作（邻居图、聚类、UMAP）均基于此 AnnData 对象。

In [ ]:
print("加载上游:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obsm keys: {list(adata.obsm.keys())}")
print(f"use_rep='{USE_REP}' -- 存在: {USE_REP in adata.obsm}")

In [ ]:
# === USE_REP 维度匹配检查 ===
# 邻居图构建前检查嵌入维度，帮助判断是否需要截断。
# Harmony/scVI 输出已是低维 → 使用全部维度。
# X_pca 可能是高维 → 建议用 N_PCS_USE_FOR_NEIGHBORS 截断前 N 维。
_embed_dim = adata.obsm[USE_REP].shape[1]
print(f"USE_REP = '{USE_REP}', 维度 = {_embed_dim}")

_n_pcs_neighbors = N_PCS_USE_FOR_NEIGHBORS  # None 或整数

if USE_REP in ("X_pca",) and _embed_dim > 30 and _n_pcs_neighbors is None:
    print(f"  ⚠️ 使用原始 PCA 全维度({_embed_dim}) 构建邻居图——包含噪音维度")
    print(f"  → 建议设 N_PCS_USE_FOR_NEIGHBORS = 30 或改用 X_pca_harmony/X_scVI")
elif _n_pcs_neighbors is not None:
    print(f"  邻居图使用前 {_n_pcs_neighbors} 维（N_PCS_USE_FOR_NEIGHBORS）")
else:
    print(f"  ✓ 嵌入已是低维({_embed_dim}D)，使用全部维度")

## 计算邻居图

在 04 选定嵌入（`USE_REP`）上构建 k 近邻图。
**为什么所有分辨率共享同一个邻居图？**
不同分辨率只改变聚类的粒度（通过 Leiden 算法的 resolution 参数），
不需要每次重新计算细胞间距离关系。共享避免了重复计算和内存浪费。

`n_pcs=None`：嵌入已是低维表示，无需再做 PCA 降维。

### N_NEIGHBORS 如何影响聚类

邻居图是 Leiden 聚类的**唯一输入**——resolution 只改变"切多细"，但哪些细胞是"邻居"由 k 决定。

| k 值 | 效果 | 适用场景 |
|---|---|---|
| 10 | 局部敏感，稀有群体可见，但噪音连接多 | 找稀有细胞类型（<1%）、transitional states |
| 15 | 平衡（默认） | 大多数分析（50k-200k cells） |
| 20-30 | 全局稳定，减少碎片化 | 数据噪音大、batch effect 残留明显 |
| 50+ | 非常粗粒度 | >200k cells 或只需大类注释 |

**经验法则**：`k ≈ sqrt(n_cells) / 3`，但最终以生物学合理性为准。

**怎么判断 k 是否合适**：
- res=0.8 只分出 3-4 个大群 → 试**减小 k**（让局部结构可见）
- res=0.8 产出 20+ 个碎片 → 试**增大 k**（让全局结构主导）
- 同一 compartment 内的亚型总是被合并 → k 可能太大

**与 resolution 的交互**：k 决定"谁和谁是邻居"，resolution 决定"切多细"。先确定合适的 k，再 sweep resolution。如果 k 不对，sweep resolution 无法补救。

In [ ]:
# 在选定嵌入上计算邻居图。
# 支持 Scalar-or-Sweep：列表模式自动对比不同 k 对分群的影响。
# 单值模式：直接计算，所有分辨率共享同一邻居图。
# _n_pcs_neighbors 来自上游维度检查 cell，None=使用全部维度。
_k_values = N_NEIGHBORS if isinstance(N_NEIGHBORS, list) else [N_NEIGHBORS]

if RECOMPUTE_NEIGHBORS or len(_k_values) > 1:
    if len(_k_values) > 1:
        # Sweep 模式：对每个 k 计算 Leiden(res=1.0)，对比 cluster 数差异
        print(f"N_NEIGHBORS sweep: {_k_values}")
        k_comparison = []
        for k in _k_values:
            sc.pp.neighbors(adata, use_rep=USE_REP, n_pcs=_n_pcs_neighbors,
                            n_neighbors=k, random_state=RANDOM_SEED)
            sc.tl.leiden(adata, resolution=1.0, key_added=f"leiden_k{k}_res1.0",
                         flavor="igraph", random_state=RANDOM_SEED)
            n_clusters = adata.obs[f"leiden_k{k}_res1.0"].nunique()
            k_comparison.append({"k": k, "n_clusters_at_res1.0": n_clusters})
        print(pd.DataFrame(k_comparison).to_string(index=False))
        # 最终使用列表最后一个 k
        sc.pp.neighbors(adata, use_rep=USE_REP, n_pcs=_n_pcs_neighbors,
                        n_neighbors=_k_values[-1], random_state=RANDOM_SEED)
    else:
        print(f"RECOMPUTE_NEIGHBORS=True，重新计算 k={_k_values[0]}")
        sc.pp.neighbors(adata, use_rep=USE_REP, n_pcs=_n_pcs_neighbors,
                        n_neighbors=_k_values[0], random_state=RANDOM_SEED)
else:
    print(f"\n===== Neighbors on {USE_REP} =====")
    sc.pp.neighbors(adata, use_rep=USE_REP, n_pcs=_n_pcs_neighbors,
                    n_neighbors=_k_values[-1], random_state=RANDOM_SEED)

print(f"Neighbor graph: {adata.obsp['connectivities'].shape}")
print(f"  n_neighbors={adata.uns['neighbors']['params']['n_neighbors']}")

## 多分辨率 Leiden 聚类

每个分辨率各做一次 Leiden 聚类。所有 `leiden_res_*` 列共存以供对比。
使用 `flavor="igraph"` 以兼容 scanpy >= 1.10。

**为什么做多个分辨率？** 没有"正确"的分辨率——不同生物学问题需要
不同粒度。粗分辨率（0.2-0.4）捕获大细胞谱系；细分辨率（1.0+）
区分亚型。PI 查看 UMAP 后根据生物学背景选择最合理的一个或几个。

In [ ]:
# 多分辨率 Leiden 聚类：每个分辨率一个 obs 列。
print(f"\n===== Leiden: {len(RESOLUTIONS)} resolutions =====")

for res in RESOLUTIONS:
    key = f"leiden_res_{res}"
    print(f"  resolution={res} -> obs['{key}']")
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=key,
        flavor="igraph",
        random_state=RANDOM_SEED,
    )
    n_clusters = adata.obs[key].nunique()
    print(f"    clusters: {n_clusters}" + (" ← 接近预期大类数" if 8 <= n_clusters <= 20 else ""))

leiden_columns = [c for c in adata.obs.columns if c.startswith("leiden_res_")]
print(f"\nLeiden columns produced: {leiden_columns}")

# 最小 cluster 大小警告
for res in RESOLUTIONS:
    key = f"leiden_res_{res}"
    sizes = adata.obs[key].value_counts()
    small = sizes[sizes < MIN_CLUSTER_SIZE]
    if len(small) > 0:
        print(f"  ⚠️ res={res}: {len(small)} 个 cluster < {MIN_CLUSTER_SIZE} 细胞: {dict(small)}")

In [ ]:
# --- 聚类稳定性分析 ---
# 子抽样 ARI：对每个分辨率，随机抽取 80% 细胞重复聚类，
# 计算子集聚类与全集聚类在子集上的 ARI。
# ARI 越高 = 分群对数据扰动越鲁棒。
if STABILITY_ENABLED:
    from sklearn.metrics import adjusted_rand_score

    print("\n===== 聚类稳定性分析 =====")
    print(f"参数: n_iter={STABILITY_N_ITER}, subsample_frac={STABILITY_SUBSAMPLE_FRAC}")

    _use_k = _k_values[-1] if isinstance(N_NEIGHBORS, list) else N_NEIGHBORS
    stability_results = []

    for res in RESOLUTIONS:
        aris = []
        full_key = f"leiden_res_{res}"
        for i in range(STABILITY_N_ITER):
            np.random.seed(RANDOM_SEED + i)
            idx = np.random.choice(
                adata.n_obs,
                int(adata.n_obs * STABILITY_SUBSAMPLE_FRAC),
                replace=False,
            )
            adata_sub = adata[idx].copy()
            # 子集需要独立的 neighbors + leiden
            sc.pp.neighbors(adata_sub, use_rep=USE_REP, n_neighbors=_use_k,
                            random_state=RANDOM_SEED)
            sc.tl.leiden(adata_sub, resolution=res, key_added="leiden_sub",
                         flavor="igraph", random_state=RANDOM_SEED)
            # ARI：子集聚类标签 vs 全集聚类标签在子集上的取值
            full_labels = adata.obs[full_key].values[idx]
            ari = adjusted_rand_score(full_labels, adata_sub.obs["leiden_sub"])
            aris.append(ari)
            del adata_sub
        stability_results.append({
            "resolution": res,
            "mean_ARI": np.mean(aris),
            "std_ARI": np.std(aris),
        })

    stability_df = pd.DataFrame(stability_results)

    # 稳定性折线图
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.errorbar(stability_df["resolution"], stability_df["mean_ARI"],
                yerr=stability_df["std_ARI"], marker="o", capsize=4)
    ax.axhline(0.9, color="green", linestyle="--", alpha=0.5,
               label="ARI=0.9（高稳定性）")
    ax.axhline(0.8, color="orange", linestyle="--", alpha=0.5,
               label="ARI=0.8（可接受）")
    ax.set_xlabel("Resolution")
    ax.set_ylabel("ARI（子抽样一致性）")
    ax.set_title("聚类稳定性：越高 = 分群对数据扰动越鲁棒")
    ax.legend()
    plt.tight_layout(); plt.show()

    print(stability_df.to_string(index=False))
else:
    print("\n聚类稳定性分析已关闭（STABILITY_ENABLED=False）。")

In [ ]:
# --- Consensus Clustering ---
# 使用多个中间分辨率构建 co-clustering 矩阵，识别稳健的细胞对
if CONSENSUS_CLUSTERING:
    from sklearn.metrics import adjusted_rand_score

    print("===== Consensus Clustering =====")
    # 使用 3 个中间分辨率做 consensus
    _consensus_res = [r for r in RESOLUTIONS if 0.4 <= r <= 1.2][:3]
    if len(_consensus_res) >= 2:
        n = adata.n_obs
        co_cluster = np.zeros((n, n), dtype=np.float32)
        for res in _consensus_res:
            key = f"leiden_res_{res}"
            labels = adata.obs[key].values
            for cl in np.unique(labels):
                idx = np.where(labels == cl)[0]
                co_cluster[np.ix_(idx, idx)] += 1
        co_cluster /= len(_consensus_res)
        # 高置信度 pair：co_cluster == 1.0（所有分辨率都共聚）
        n_robust_pairs = int((co_cluster == 1.0).sum() - n) // 2  # 减去对角线
        n_total_pairs = n * (n - 1) // 2
        print(f"  Robust pairs（所有 {len(_consensus_res)} 分辨率共聚）: {n_robust_pairs:,} / {n_total_pairs:,} ({100*n_robust_pairs/n_total_pairs:.1f}%)")
        print(f"  ⚠️ 注意：consensus clustering 在大数据集上内存密集（n² 矩阵）")
        del co_cluster
    else:
        print("  跳过：需要至少 2 个在 [0.4, 1.2] 范围内的分辨率")


In [ ]:
# --- Marker 基因预览：每个分辨率的 top DEG（快速 sanity check）---
# 如果所有 cluster 的 top gene 都是核糖体/线粒体 → HVG 有问题，回 03 调整
if MARKER_PREVIEW_ENABLED:
    print("\n" + "=" * 60)
    print("Marker 预览：每个分辨率的 top DEG（快速 sanity check）")
    print("如果所有 cluster 的 top gene 都是核糖体/线粒体 -> HVG 有问题，回 03")
    print("=" * 60)

    for res in RESOLUTIONS:
        key = f"leiden_res_{res}"
        n_clusters = adata.obs[key].nunique()
        sc.tl.rank_genes_groups(
            adata, groupby=key, method="wilcoxon",
            n_genes=MARKER_PREVIEW_N_GENES, use_raw=False,
        )

        print(f"\n--- Resolution {res} ({n_clusters} clusters) ---")
        # 紧凑表格：每行一个 cluster，列出 top N 基因
        result = adata.uns["rank_genes_groups"]
        groups = result["names"].dtype.names
        for group in groups[:min(10, len(groups))]:  # 最多显示 10 个 cluster
            genes = [result["names"][group][i]
                     for i in range(MARKER_PREVIEW_N_GENES)]
            scores = [f"{result['scores'][group][i]:.1f}"
                      for i in range(MARKER_PREVIEW_N_GENES)]
            print(f"  Cluster {group}: {', '.join(genes)} "
                  f"(scores: {', '.join(scores)})")
        if len(groups) > 10:
            print(f"  ... 共 {len(groups)} clusters，仅显示前 10")

    # 清理——避免最后一个分辨率的 DEG 结果被误当做"最终" DEG 结果
    del adata.uns["rank_genes_groups"]
    print("\nrank_genes_groups 已清理（仅预览用，非最终 DEG 结果）。")
else:
    print("\nMarker 预览已关闭（MARKER_PREVIEW_ENABLED=False）。")

## 聚类指标对比——辅助选择最佳分辨率

对每个分辨率计算聚类质量指标，与可视化结果互为补充。

**直接遍历分辨率列表**——没有回调、没有封装。学生逐行可读。

**指标说明**（数据允许时计算）：
- **silhouette score**（轮廓系数）：衡量簇的紧密程度与分离度。值越接近 1 越好。
  在 PCA 空间上计算，反映嵌入空间中簇的结构质量。
- **ARI**（调整兰德指数）：与已知标签的一致性。需要 `obs` 中存在参考标签列。

**如何用指标辅助决策**：
- silhouette 随分辨率升高通常会缓慢下降——这是正常的（更细的簇边界更模糊）。
  重点看下降趋势中是否存在"拐点"（分辨率增加但 silhouette 骤降），而非绝对值。
- ARI 仅在存在参考标签时有意义。
- **最终决策以 UMAP 可视化为主，指标为参考**——生物学的分群合理性不能仅由数值指标决定。

对比报告写入 `results/figures/sweep_05/sweep_report.md`。

In [ ]:
# 显式 for 循环：遍历各分辨率，运行 Leiden 并计算聚类指标。
# 每个分辨率在独立拷贝上运行，避免列名冲突。
print("\n===== 显式遍历分辨率 + clustering_metrics =====\n")

results = []
# 单次拷贝复用：clustering_metrics 只读 obsm/obs，不修改 adata，
# 因此可在同一份拷贝上重跑不同 resolution 的 leiden（各分辨率写不同 obs 列），
# 避免 8 个分辨率各 deep copy 一次导致的 8x 内存峰值。
_adata_metrics = adata.copy()

for res in RESOLUTIONS:
    print(f"--- resolution={res} ---")

    key = f"leiden_res_{res}"
    sc.tl.leiden(
        _adata_metrics,
        resolution=res,
        key_added=key,
        flavor="igraph",
        random_state=RANDOM_SEED,
    )

    # 直接调用聚类指标函数——显式传入 cluster_key=key 避免 auto-detect 误选其他 leiden 列
    m = clustering_metrics(_adata_metrics, cluster_key=key)
    results.append({"resolution": res, **m})

    n_cl = _adata_metrics.obs[key].nunique()
    metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in m.items()
                            if isinstance(v, float) and not np.isnan(v))
    print(f"  簇数: {n_cl}  指标: {metrics_str}")

del _adata_metrics

# 收集为 DataFrame 对比表
sweep_df = pd.DataFrame(results)
os.makedirs("results/figures/sweep_05", exist_ok=True)

# 写 Markdown 报告
lines = ["# 05 聚类对比报告\n",
         f"**{len(RESOLUTIONS)} 个分辨率** 已评估。\n",
         "## 指标表\n"]
lines.append("| " + " | ".join(sweep_df.columns) + " |")
lines.append("|" + "|".join(" --- " for _ in sweep_df.columns) + "|")
for _, row in sweep_df.iterrows():
    vals = []
    for col in sweep_df.columns:
        v = row[col]
        if isinstance(v, float):
            vals.append(f"{v:.4f}" if not np.isnan(v) else "N/A")
        else:
            vals.append(str(v))
    lines.append("| " + " | ".join(vals) + " |")
with open("results/figures/sweep_05/sweep_report.md", "w") as f:
    f.write("\n".join(lines) + "\n")

# 绘制指标随分辨率变化的折线图——辅助 PI 识别拐点
metric_cols = [c for c in sweep_df.columns
               if c != "resolution" and not sweep_df[c].isna().all()]
if metric_cols:
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(5 * len(metric_cols), 4))
    if len(metric_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, metric_cols):
        ax.plot(sweep_df["resolution"], sweep_df[col], "o-", color="#2c7bb6", markersize=6)
        ax.set_xlabel("resolution")
        ax.set_ylabel(col)
        ax.set_title(f"{col} vs resolution")
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig_path = "results/figures/sweep_05/metrics_vs_resolution.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"\n指标-分辨率折线图: {fig_path}")
    plt.close("all")

# 展示对比表
print("\n聚类指标对比表:")
try:
    from IPython.display import display as ipy_display
    ipy_display(sweep_df)
except ImportError:
    print(sweep_df)

print("\n对比报告: results/figures/sweep_05/sweep_report.md")
adata.uns["05_sweep_v1"] = {
    "resolutions_swept": RESOLUTIONS,
    "scorer": "clustering_metrics",
    "report_dir": "results/figures/sweep_05",
    "timestamp": datetime.datetime.now().isoformat(),
}

## 扩展槽——其他聚类方法

任何写出 `obs["{method}_clusters"]` 的聚类方法均可与 `leiden_res_*` 列共存
并以同样方式参与对比。

**ACDC**（已注释）：全局搜索最优分区。**不**是默认依赖（在 GCPL 数据上
运行太慢）。PI 安装后取消注释，新列自动进入 06。

**新方法的通用模式**：写 `obs["{method}_clusters"]` = labels，
然后加入显式 for 循环的 candidates 列表或单独对比——
与 04 嵌入同样的并行槽位约定，零框架改动。

In [ ]:
# # === ACDC clustering (commented out -- NOT a default dependency) ===
# # PREREQUISITE: pip install acdc_py
# # Enable by removing comments below.
#
# # import acdc_py  # 包名/导入名以 PyPI 实际为准，启用前先确认；ACDC 非默认依赖
# # # ACDC searches for an optimal partition.
# # acdc_result = ACDC.ACDC(adata, ...)
# # adata.obs["acdc_clusters"] = acdc_result.labels
# # print(f"ACDC: {adata.obs['acdc_clusters'].nunique()} clusters")
#
# print("ACDC cell is commented out. "
#       "Uncomment when acdc_py is installed and suitable for this dataset.")
#
# # === Add any future method here ===
# # Pattern: write adata.obs["{method}_clusters"] = labels
# # Then add to sweep candidates or compare standalone.
# # Example: adata.obs["foocluster_clusters"] = foo_cluster.fit_predict(
# #     adata.obsm[USE_REP])

## 各分辨率 UMAP 可视化

**按分辨率逐个着色 UMAP**——每个分辨率算完即出图。
PI 可以跑一个分辨率看一个，判断该分辨率的聚类是否合理，再决定最终采纳哪个。

**怎么看 UMAP 判断分群质量**：
- 同一簇在 UMAP 上应该聚在一起（簇内紧凑）
- 不同簇之间应该有清晰的边界或过渡
- 如果某个分辨率把一个明显的谱系拆成多块 → 过分（分辨率太高）
- 如果某个分辨率把明显不同的细胞群合并 → 欠分（分辨率太低）
- 在 UMAP 上看簇的分布，结合下方的群大小分布图，综合判断

In [ ]:
# 从已有邻居图计算 UMAP（所有分辨率共享同一嵌入）。
# 然后逐个分辨率着色，每个分辨率即时出图，方便 PI 对比选择。
print("\n===== UMAP per resolution =====")

sc.tl.umap(adata, random_state=RANDOM_SEED)

leiden_cols = sorted(
    [c for c in adata.obs.columns if c.startswith("leiden_res_")],
    key=lambda x: float(x.split("_")[-1]),
)

for col in leiden_cols:
    n_clusters = adata.obs[col].nunique()
    res_str = col.split("_")[-1]
    print(f"\n--- resolution={res_str}: {n_clusters} clusters ---")

    sc.pl.umap(
        adata, color=col,
        title=f"Leiden res={res_str}（{n_clusters} 群）",
        legend_loc="on data" if n_clusters <= 10 else "right margin",
        frameon=False,
        save=f"_05_{col}.png",
    )
    # 从 scanpy 默认 figures/ 目录移到 results/figures/
    src = f"figures/umap_05_{col}.png"
    dst = f"results/figures/05_umap_{col}.png"
    if os.path.exists(src):
        os.rename(src, dst)
        print(f"    图已保存: {dst}")
    else:
        print(f"    注意: 图未在默认位置 {src} 找到")
    plt.close("all")

print(f"\nUMAP 出图完成，共 {len(leiden_cols)} 个分辨率。")

## 群大小分布——辅助判断分辨率合理性

除了 UMAP 可视化，各分辨率下的群大小分布也是选择分群方案的重要参考：

- **如果存在极大群和极小群并存**（一个群占了 50% 以上细胞，另一群只有几十个）→ 可能分辨率不合适
- **如果群大小相对均匀** → 分群粒度与数据结构匹配较好
- **极小的群**（< 1% 总细胞数）可能是噪声或稀有细胞亚型——需要结合生物学背景判断

下方的条形图展示每个分辨率下各群的细胞数分布。

In [ ]:
# 各分辨率下的群大小分布——帮助 PI 判断分群是否合理。
print("\n===== 群大小分布 =====")

n_res = len(RESOLUTIONS)
n_cols = 3
n_rows = (n_res + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
if n_rows == 1 and n_cols == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for i, res in enumerate(RESOLUTIONS):
    key = f"leiden_res_{res}"
    cluster_sizes = adata.obs[key].value_counts().sort_index()
    n_clusters = len(cluster_sizes)

    ax = axes[i]
    colors = plt.cm.tab20(np.linspace(0, 1, n_clusters))
    ax.bar(range(n_clusters), cluster_sizes.values, color=colors)
    ax.set_title(f"resolution={res}（{n_clusters} 群）")
    ax.set_xlabel("群编号")
    ax.set_ylabel("细胞数")
    ax.set_xticks(range(n_clusters))
    ax.tick_params(axis="x", labelsize=8)

# 隐藏多余的子图
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig_path = "results/figures/05_cluster_sizes.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"群大小分布图保存至: {fig_path}")
plt.close("all")

## 分辨率选择建议

以下基于预期 cluster 数范围做**启发式推荐**。这不是自动选择——最终决策以 UMAP 目测为准。

**推荐逻辑**：
- 胃黏膜预期 ~5 大类（上皮/免疫/间质/内分泌/增殖），对应 5-20 个 cluster 的分辨率
- 倾向于选择该窗口内**稍高**的分辨率（宁可稍过聚类，后续通过 dendrogram 合并）
- 超过 20 个 cluster 的分辨率适合"过聚类探索"策略——配合 dendrogram 识别可合并的姐妹 cluster

**注意事项**：
- 指标仅做参考，最终决策以 UMAP 目测为准
- 关注同一簇是否紧凑、不同簇边界是否清晰、已知 marker 分布是否合理
- 如有稳定性数据，优先选择高 ARI 的分辨率

In [ ]:
# === 分辨率选择建议 ===
# 基于预期 cluster 数范围做启发式推荐。这不是自动选择——
# 最终决策以 UMAP 目测为准，指标和推荐仅做参考。
print("===== 分辨率选择建议 =====\n")

_n_clusters = {res: adata.obs[f"leiden_res_{res}"].nunique() for res in RESOLUTIONS}
for res in RESOLUTIONS:
    print(f"  res={res}: {_n_clusters[res]} clusters")

# 启发式推荐：基于预期 cluster 数范围
# 胃黏膜 ~5 大类（上皮/免疫/间质/内分泌/增殖）
_expected_major = 5
# 含主要亚型
_expected_fine = 20

_annot_candidates = [r for r in RESOLUTIONS
                     if _expected_major <= _n_clusters[r] <= _expected_fine]
_explore_candidates = [r for r in RESOLUTIONS if _n_clusters[r] > _expected_fine]

print(f"\n  预期大类数: ~{_expected_major}（上皮/免疫/间质/内分泌/增殖）")
print(f"  预期亚型数: ~{_expected_fine}（含主要亚型）")
print()

if _annot_candidates:
    print(f"  📌 注释推荐（{_expected_major}-{_expected_fine} 簇）: res={_annot_candidates}")
    _rec = _annot_candidates[-1]  # 倾向于稍高分辨率
    print(f'     stage 06 设置: CLUSTER_KEY = "leiden_res_{_rec}"')
else:
    print(f"  ⚠️ 无分辨率产出 {_expected_major}-{_expected_fine} 个簇——调整 RESOLUTIONS 范围")
    _rec = RESOLUTIONS[len(RESOLUTIONS)//2]

if _explore_candidates:
    print(f"  🔬 过聚类探索: res={_explore_candidates[:2]}")
    print(f"     → 配合 dendrogram 合并策略，适合发现新亚型/transitional states")

# 稳定性提示（如有稳定性数据）
if STABILITY_ENABLED and "stability_df" in dir():
    _stable_candidates = stability_df[stability_df["mean_ARI"] >= 0.85]["resolution"].tolist()
    if _stable_candidates:
        print(f"\n  高稳定性分辨率（ARI ≥ 0.85）: {_stable_candidates}")

print(f"\n  ⚠️ 最终决策以 UMAP 目测为准——指标仅做参考")
print(f"     关注: 同一簇是否紧凑、不同簇边界是否清晰、已知 marker 分布是否合理")

In [ ]:
# Doublet cluster 筛查：per-cluster mean(doublet_score)
# 如果某 cluster 平均 doublet score > 2× 全局均值 → 疑似 doublet cluster
if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
    # 使用 PI 选定的分辨率（或默认最后一个 RESOLUTIONS）
    _chosen_res = OVER_CLUSTERING_RES if OVER_CLUSTERING_RES else RESOLUTIONS[-1]
    _chosen_key = f"leiden_res_{_chosen_res}"
    if _chosen_key in adata.obs.columns:
        global_mean = adata.obs["doublet_score"].mean()
        cluster_doublet = adata.obs.groupby(_chosen_key)["doublet_score"].mean()
        suspect = cluster_doublet[cluster_doublet > 2 * global_mean]
        print(f"===== Doublet Cluster 筛查（resolution={_chosen_res}）=====")
        print(f"全局 mean doublet_score: {global_mean:.4f}")
        if len(suspect) > 0:
            print(f"⚠️ 疑似 doublet cluster（mean > 2×全局均值）:")
            for cl, score in suspect.items():
                n_cells = (adata.obs[_chosen_key] == cl).sum()
                print(f"  Cluster {cl}: mean_doublet={score:.4f} ({score/global_mean:.1f}×), n_cells={n_cells}")
            print("  → 建议在 06 注释时特别关注这些 cluster，可能需要移除")
        else:
            print("✓ 无疑似 doublet cluster")
else:
    print("跳过 doublet cluster 筛查：doublet_score 列不可用")


In [ ]:
# --- 沉默 cluster 检测：top marker 与邻近 cluster 高度重叠的 cluster 可能是噪声或过分裂 ---
if MARKER_PREVIEW_ENABLED:
    _last_res = RESOLUTIONS[-1]
    _last_key = f"leiden_res_{_last_res}"
    sc.tl.rank_genes_groups(adata, groupby=_last_key, method="wilcoxon", n_genes=5, use_raw=False)
    result = adata.uns["rank_genes_groups"]
    groups = list(result["names"].dtype.names)
    # 每个 cluster 的 top5 基因集
    cluster_markers = {}
    for g in groups:
        cluster_markers[g] = set(result["names"][g][:5])
    # 对比邻近 cluster 的 marker 重叠
    silent_pairs = []
    for i, g1 in enumerate(groups):
        for g2 in groups[i+1:]:
            jaccard = len(cluster_markers[g1] & cluster_markers[g2]) / max(len(cluster_markers[g1] | cluster_markers[g2]), 1)
            if jaccard > 0.4:
                silent_pairs.append((g1, g2, jaccard))
    if silent_pairs:
        print(f"⚠️ {len(silent_pairs)} 对 cluster 的 top-5 marker 高度重叠（Jaccard > 0.4）:")
        for g1, g2, j in sorted(silent_pairs, key=lambda x: -x[2])[:5]:
            print(f"  Cluster {g1} vs {g2}: Jaccard={j:.2f} → 考虑合并")
    else:
        print("✓ 无沉默 cluster 对（各 cluster marker 特异）")
    del adata.uns["rank_genes_groups"]


## Cluster 层级结构（Dendrogram）

Dendrogram 展示各 cluster 在嵌入空间中的层级相似性。相邻分支上的 cluster
表达谱相近，是合并的首选候选——尤其在高分辨率（过聚类）策略下，
通过 dendrogram 识别可合并的姐妹 cluster 是注释前的关键步骤。


In [ ]:
# Cluster 层级结构（dendrogram）——辅助判断哪些 cluster 可以合并
_chosen_res = OVER_CLUSTERING_RES if OVER_CLUSTERING_RES else RESOLUTIONS[-1]
_chosen_key = f"leiden_res_{_chosen_res}"
if _chosen_key in adata.obs.columns:
    print(f"===== Dendrogram（resolution={_chosen_res}）=====")
    sc.tl.dendrogram(adata, groupby=_chosen_key, use_rep=USE_REP)
    sc.pl.dendrogram(adata, groupby=_chosen_key)
    plt.tight_layout(); plt.show()
    print("相邻分支上的 cluster 是合并的首选候选")


## QC 残留诊断

即使 01 已做 QC 过滤，仍可能有低质量细胞残留在数据中并形成独立 cluster。
本诊断检查每个 cluster 的 QC 指标分布（pct_mt、doublet_score、log_complexity），
标记中位值显著高于全局水平的 cluster——提示可能需要收紧 01 的 QC 阈值。


In [ ]:
# QC 残留诊断：检查是否有 cluster 被 QC 遗漏的低质量细胞主导
# 如果某 cluster 的中位 pct_mt / doublet_score 是 outlier → 提示回 01 收紧 QC
_chosen_res = OVER_CLUSTERING_RES if OVER_CLUSTERING_RES else RESOLUTIONS[-1]
_chosen_key = f"leiden_res_{_chosen_res}"
if _chosen_key in adata.obs.columns:
    print(f"===== QC 残留诊断（resolution={_chosen_res}）=====")
    qc_metrics = ["pct_counts_mt"]
    if "doublet_score" in adata.obs.columns:
        qc_metrics.append("doublet_score")
    if "log_complexity" in adata.obs.columns:
        qc_metrics.append("log_complexity")

    fig, axes = plt.subplots(1, len(qc_metrics), figsize=(6*len(qc_metrics), 5))
    if len(qc_metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, qc_metrics):
        sc.pl.violin(adata, keys=metric, groupby=_chosen_key, rotation=45, ax=ax, show=False)
        ax.set_title(f"{metric} per cluster")
        # 标记 outlier cluster（中位值 > 全局中位 + 2*MAD）
        global_med = adata.obs[metric].median()
        global_mad = median_abs_deviation(adata.obs[metric].dropna(), nan_policy="omit")
        # 用颜色标记 outlier——scanpy violin plot 的 patch collection 索引对应 sorted unique groups
        clusters_sorted = sorted(adata.obs[_chosen_key].unique())
        for i, cl in enumerate(clusters_sorted):
            cl_med = adata.obs.loc[adata.obs[_chosen_key] == cl, metric].median()
            if cl_med > global_med + 2 * global_mad:
                # 尝试修改 violin patch 颜色（兼容 scanpy >= 1.10）
                for collection in ax.collections:
                    if hasattr(collection, "set_facecolor"):
                        try:
                            face_colors = collection.get_facecolors()
                            if len(face_colors) > i:
                                face_colors[i] = [1.0, 0.0, 0.0, 0.6]
                                collection.set_facecolors(face_colors)
                        except (IndexError, ValueError):
                            pass
    plt.tight_layout(); plt.show()

    # 文字摘要
    for metric in qc_metrics:
        global_med = adata.obs[metric].median()
        global_mad = median_abs_deviation(adata.obs[metric].dropna(), nan_policy="omit")
        outlier_clusters = []
        for cl in sorted(adata.obs[_chosen_key].unique()):
            cl_med = adata.obs.loc[adata.obs[_chosen_key] == cl, metric].median()
            if cl_med > global_med + 2 * global_mad:
                outlier_clusters.append(f"{cl}(med={cl_med:.2f})")
        if outlier_clusters:
            print(f"⚠️ {metric} outlier clusters: {', '.join(outlier_clusters)}")
            print(f"  全局 median={global_med:.2f}, threshold={global_med + 2*global_mad:.2f}")
        else:
            print(f"✓ {metric}: 无 outlier cluster")


## 跨数据集 cluster 归属偏差

即使 04 做了 batch correction，仍可能有某些 cluster 被单一数据集主导。
这个诊断检查每个 cluster 的细胞来源分布：

- **>80% 来自单一数据集**：需判断是真实的生物学差异（某数据集含独特组织区域）
  还是 batch correction 不完全（该 cluster 本质上是数据集批次效应）
- **分布均衡**：说明 batch correction 在此分辨率下有效，跨数据集的细胞类型具有可比性

如果发现严重偏差：回到 04 检查 UMAP batch 着色，判断是否需要调整校正参数。

In [ ]:
# === 跨数据集 cluster 归属偏差检测 ===
# 检查各 cluster 是否被单一数据集主导（>80% 细胞来自同一来源）。
# 这帮助区分"真实的生物学差异"与"batch correction 不完全"。
# 如果发现严重偏差：回 04 检查 UMAP batch 着色，判断是否需要调整校正参数。
_batch_col = None
# 尝试从 04 的 uns 中读取 BATCH_KEY
if "harmony_v1" in adata.uns and "batch_key" in adata.uns["harmony_v1"]:
    _batch_col = adata.uns["harmony_v1"]["batch_key"]
elif "scvi_v1" in adata.uns and "batch_key" in adata.uns["scvi_v1"]:
    _batch_col = adata.uns["scvi_v1"]["batch_key"]
else:
    # 尝试常见 batch 列名
    for cand in ["source_dataset", "batch", "sample_id"]:
        if cand in adata.obs.columns:
            _batch_col = cand
            break

if _batch_col and _batch_col in adata.obs.columns:
    _diag_res = RESOLUTIONS[len(RESOLUTIONS) // 2]
    _diag_key = f"leiden_res_{_diag_res}"

    if _diag_key in adata.obs.columns:
        print(f"===== 跨数据集 cluster 归属偏差（{_diag_key}, batch={_batch_col}）=====\n")

        _ct = pd.crosstab(adata.obs[_diag_key], adata.obs[_batch_col], normalize="index")
        _dominant_pct = _ct.max(axis=1)
        _biased = _dominant_pct[_dominant_pct > 0.8]

        if len(_biased) > 0:
            print(f"  ⚠️ 以下 cluster 被单一数据集主导（>80% 细胞来自同一来源）:")
            for cl in _biased.index:
                _dom_dataset = _ct.loc[cl].idxmax()
                _pct = _dominant_pct[cl]
                _n = int((adata.obs[_diag_key] == cl).sum())
                print(f"    Cluster {cl} ({_n} cells): {_pct:.0%} 来自 {_dom_dataset}")
            print()
            print("  → 可能原因:")
            print("    (a) 真实的数据集特异群（如某数据集含独特组织区域）——正常")
            print("    (b) batch correction 不完全——检查 04 UMAP batch 着色")
            print("    (c) 数据集间细胞类型组成差异——正常但需注意下游统计")
        else:
            print(f"  ✓ 所有 cluster 跨数据集分布均衡（无 >80% 单源主导）")
else:
    print("未找到 batch 列，跳过跨数据集偏差检测")

## Sanity Marker 验证

用已知生物学 marker 验证嵌入空间的基本分离格局是否正确。这不是正式注释，
只是快速 sanity check——如果 EPCAM（上皮）、PTPRC/CD45（免疫）、VIM（间质）
三大 compartment 在 UMAP 上没有明显分离，说明上游嵌入或 HVG 选择有问题。


In [ ]:
# --- Cluster 纯度：当有先验注释时，评估每个 cluster 的细胞类型一致性 ---
_annot_cols = [c for c in adata.obs.columns if c.startswith("cell_type_original_")]
if _annot_cols:
    _annot_col = _annot_cols[0]
    _valid = adata.obs[_annot_col].notna()
    if _valid.sum() > 50:
        _chosen_res = OVER_CLUSTERING_RES if OVER_CLUSTERING_RES else RESOLUTIONS[-1]
        _chosen_key = f"leiden_res_{_chosen_res}"
        if _chosen_key in adata.obs.columns:
            print(f"===== Cluster 纯度（基于 {_annot_col}）=====")
            purity_data = []
            for cl in sorted(adata.obs[_chosen_key].unique()):
                mask = (adata.obs[_chosen_key] == cl) & _valid
                if mask.sum() == 0:
                    continue
                counts = adata.obs.loc[mask, _annot_col].value_counts()
                top_type = counts.index[0]
                purity = counts.iloc[0] / mask.sum()
                purity_data.append({"cluster": cl, "dominant_type": top_type,
                                    "purity": purity, "n_cells": mask.sum()})
            purity_df = pd.DataFrame(purity_data)
            low_purity = purity_df[purity_df["purity"] < 0.6]
            if len(low_purity) > 0:
                print(f"⚠️ {len(low_purity)} 个 cluster 纯度 < 60%（混合 cluster）:")
                print(low_purity.to_string(index=False))
            else:
                print(f"✓ 全部 cluster 纯度 ≥ 60%")
            print(f"\n平均纯度: {purity_df['purity'].mean():.1%}")


In [ ]:
# Sanity Marker 验证：已知生物学分离格局是否正确？
# 如果 EPCAM+/PTPRC+/VIM+ 三大 compartment 没有分开 → 上游有问题
if SANITY_MARKERS:
    available_markers = {name: gene for name, gene in SANITY_MARKERS.items() if gene in adata.var_names}
    missing = {name: gene for name, gene in SANITY_MARKERS.items() if gene not in adata.var_names}
    if missing:
        print(f"⚠️ 部分 sanity marker 不在数据中: {missing}")
    if available_markers:
        print(f"===== Sanity Marker 验证 =====")
        genes_to_plot = list(available_markers.values())
        sc.pl.umap(adata, color=genes_to_plot, ncols=min(len(genes_to_plot), 4),
                   use_raw=False, show=True)
        print("检查：各 marker 是否形成明显分离的区域？")
        print("  EPCAM+ = 上皮 | PTPRC(CD45)+ = 免疫 | VIM+ = 间质 | MKI67+ = 增殖")
        print("  如果全部混在一起 → 可能 04 过校正或 03 HVG 有问题")


## Compartment 归属诊断与分层聚类建议

**为什么单一 resolution 不够？**

上皮 compartment 内部存在连续的 SPEM/IM 渐变——需要高 resolution（1.0-2.0）
才能分离出壁细胞、主细胞、颈黏液细胞、表面黏液细胞、SPEM、IM、异型增生的亚型梯度。
但在同样高 resolution 下，免疫 compartment 会过度碎片化（T/B/Myeloid 亚型被拆成数十个碎片），
增加下游注释负担且统计功效下降。

**分层聚类（hierarchical clustering）**是资深专家的标准做法：
1. 先在中等分辨率（0.6-1.0）粗分 compartment（上皮/免疫/间质/增殖）
2. 再对感兴趣的 compartment（通常是上皮）取子集（subset），用更高分辨率重聚类
3. 各 compartment 独立注释后再合并结果

下方细胞自动识别各 cluster 的主导 compartment 并给出 subset 重聚类建议。

In [ ]:
# === Compartment 归属诊断 + 分层聚类建议 ===
# 单一 resolution 不能同时满足所有 compartment——上皮内部的 SPEM/IM 渐变需要
# 高 resolution 才能分离，而免疫 compartment 在同样高 res 下会过度碎片化。
# 分层聚类（先粗分 compartment → 再对 subset 重聚类）是资深专家的标准做法。
# 本 cell 用 sanity marker 的表达模式帮助判断哪个 compartment 最需要 subset 重聚类。
if SANITY_MARKERS:
    # 选择一个中等分辨率做诊断（RESOLUTIONS 中间偏高位置）
    _diag_res = RESOLUTIONS[len(RESOLUTIONS) // 2]  # 约 0.8 或 1.0
    _diag_key = f"leiden_res_{_diag_res}"

    if _diag_key in adata.obs.columns:
        print(f"===== Compartment 归属诊断（基于 {_diag_key}）=====\n")

        # 计算每个 cluster 的各 compartment marker 平均表达
        _available = {name: gene for name, gene in SANITY_MARKERS.items()
                      if gene in adata.var_names}

        if _available:
            _profiles = []
            for cl in sorted(adata.obs[_diag_key].unique(), key=lambda x: int(x)):
                _mask = adata.obs[_diag_key] == cl
                _row = {"cluster": cl, "n_cells": int(_mask.sum())}
                for name, gene in _available.items():
                    _expr = adata[_mask, gene].X
                    if sp.issparse(_expr):
                        _expr = _expr.toarray()
                    _row[name] = float(_expr.mean())
                _profiles.append(_row)

            _prof_df = pd.DataFrame(_profiles)

            # 判断每个 cluster 的主导 compartment（marker 平均表达最高者）
            _marker_cols = [c for c in _prof_df.columns if c not in ["cluster", "n_cells"]]
            _prof_df["dominant"] = _prof_df[_marker_cols].idxmax(axis=1)

            # 分组输出
            _compartments = {}
            for _, row in _prof_df.iterrows():
                comp = row["dominant"]
                if comp not in _compartments:
                    _compartments[comp] = []
                _compartments[comp].append(str(row["cluster"]))

            for comp, clusters in sorted(_compartments.items()):
                print(f"  {comp} 为主: clusters {', '.join(clusters)}")

            # 检测混合 compartment cluster（多个 marker 都高表达 → 可能欠分）
            print()
            _mixed = []
            for _, row in _prof_df.iterrows():
                _vals = row[_marker_cols]
                _high = _vals[_vals > _vals.median() + _vals.std()].index.tolist()
                if len(_high) >= 2:
                    _mixed.append((row["cluster"], _high))

            if _mixed:
                print("  ⚠️ 混合 compartment cluster（可能欠分）:")
                for cl, comps in _mixed:
                    print(f"    Cluster {cl}: {' + '.join(comps)} 均高表达")
            else:
                print("  ✓ 无明显混合 compartment cluster")

            # 分层聚类建议
            print(f"\n--- 分层聚类建议 ---")
            if "Epithelial" in _compartments:
                _epi_clusters = _compartments["Epithelial"]
                _epi_n = _prof_df[_prof_df["cluster"].isin(_epi_clusters)]["n_cells"].sum()
                print(f"  上皮 subset (clusters {', '.join(_epi_clusters)}, ~{_epi_n:,} cells):")
                print(f"    → 建议在 stage 06c 中单独取子集重聚类（res=1.0-2.0）")
                print(f"    → 目标：分离壁细胞/主细胞/颈黏液/表面黏液/SPEM/IM/异型增生")
            if "Immune" in _compartments:
                _imm_clusters = _compartments["Immune"]
                _imm_n = _prof_df[_prof_df["cluster"].isin(_imm_clusters)]["n_cells"].sum()
                print(f"  免疫 subset (clusters {', '.join(_imm_clusters)}, ~{_imm_n:,} cells):")
                print(f"    → 如需细分 T/B/Myeloid 亚型，同样建议子集重聚类")
                print(f"    → 如只需大类注释（T/B/Myeloid/NK），当前分辨率可能够用")
            if "Stromal" in _compartments:
                _stro_clusters = _compartments["Stromal"]
                _stro_n = _prof_df[_prof_df["cluster"].isin(_stro_clusters)]["n_cells"].sum()
                print(f"  间质 subset (clusters {', '.join(_stro_clusters)}, ~{_stro_n:,} cells):")
                print(f"    → 如需分离成纤维/平滑肌/周细胞/间皮亚型，建议子集重聚类")

## 运行元数据

版控键记录聚类参数与追溯链，供后续审计查询。

In [ ]:
# 记录聚类运行元数据——包含参数、版本与追溯链。
print("\n===== Run metadata =====")

adata.uns["leiden_v1"] = {
    "use_rep": USE_REP,
    "resolutions": RESOLUTIONS,
    "flavor": "igraph",
    "n_neighbors": adata.uns["neighbors"]["params"]["n_neighbors"],
    "timestamp": datetime.datetime.now().isoformat(),
}

# 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 编号命名一致）
adata.uns["stage"] = "05_clustered"     # 本 stage 标识
adata.uns["version"] = "v1"             # 与 OUTPUT_PATH 版本号一致
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"    # PI 审查后改为 "promoted"

# 各分辨率的簇数汇总
cluster_summary = {}
for col in leiden_cols:
    cluster_summary[col] = int(adata.obs[col].nunique())
adata.uns["leiden_v1"]["cluster_counts"] = cluster_summary

print("Clusters per resolution:")
for col, n in cluster_summary.items():
    print(f"  {col}: {n}")
print(f"status: {adata.uns['status']}")

In [ ]:
# 写入前自检——确保 X 保持稀疏 float32，防止意外 densify 导致内存暴涨。
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 意外退化: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("自检通过: X 为稀疏 CSR float32。")

In [ ]:
# 写出 checkpoint——lzf 压缩兼顾速度与空间，保留稀疏 CSR 布局。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写入: {OUTPUT_PATH}")

import os
assert os.path.exists(OUTPUT_PATH), f"文件未找到: {OUTPUT_PATH}"
print(f"验证通过: {OUTPUT_PATH}（{os.path.getsize(OUTPUT_PATH):,} bytes）")

In [ ]:
# 跨 stage 边界释放内存。
# 如果 Jupyter 内核会话中接着跑下一 stage（06），
# 这一步避免两个 stage 的 AnnData 同时驻留内存导致 OOM。
del adata
import gc
gc.collect()
print("内存已释放。")